# Prueba de estrés de featurización — §11.4-bis

## Modo actual: DIAGNÓSTICO DE INYECCIÓN

Antes de concluir sobre suficiencia del oráculo, verificar que el canal de
confound está realmente abierto. Dos correlaciones a medir:

- **c1** = `corr(n_JL_fill_prev, depth_tras_inyectar)` — ¿la inyección crea profundidad?
- **c2** = `corr(n_JL_fill_prev, depth_en_decision)` — ¿esa profundidad persiste hasta la decisión?

Si c1 >> 0 y c2 ≈ 0: la inyección funciona pero el fill la borra en un paso.  
Si c1 ≈ 0: el generador no está abriendo el canal.  
Si c2 >> 0: el canal está vivo en la decisión y la regresión tiene algo que medir.

**Correcciones activas:**
- Bug 1: `base_val` en controles del oráculo = `extended_board_value` (no depth-sensitive).
- Bug 2: logging lag-1 para diagnosticar borrado; no se cambia correlación a contemporánea
  (eso fabricaría un confound de estructura distinta al del humano real).

In [ ]:
# ============================================================
# CONFIGURACIÓN
# ============================================================
REPO_URL = "https://github.com/yudocyudoc/tetris.git"

N_GAMES    = 50    # diagnóstico: 50 partidas es suficiente para ver las correlaciones
MAX_PIECES = 500
TAU        = 10.0
BVW        = 0.5
K          = 5
ALPHA_DEPTH = 0.4
N_WORKERS  = 4     # Colab Plus. Cambiar a 1 si hay deadlock.

# Diagnóstico: solo intensidad máxima para maximizar señal
P_BAG_FILL_LEVELS = [1.0]

OUT_DIR_BASE = "/content/tetris/colab/results_colab"

print(f"MODO: diagnóstico de inyección (p_bag_fill={P_BAG_FILL_LEVELS})")
print(f"N_GAMES={N_GAMES}, N_WORKERS={N_WORKERS}")

In [ ]:
# ============================================================
# CLONAR REPO E INSTALAR DEPENDENCIAS
# ============================================================
import os

if not os.path.exists("/content/tetris"):
    !git clone {REPO_URL} /content/tetris
else:
    !cd /content/tetris && git pull

!pip install -q pygame pandas pyarrow numpy scipy matplotlib tabulate statsmodels

print("Repo listo.")

In [ ]:
import os, json, subprocess, sys
import numpy as np
import pandas as pd

CONF_DIR = "/content/tetris/analysis/confound_floor"
os.makedirs(OUT_DIR_BASE, exist_ok=True)

# Verificar que el script de estres existe
STRESS_SCRIPT = os.path.join(CONF_DIR, "confound_floor_stress_t2.py")
assert os.path.exists(STRESS_SCRIPT), f"No se encontró {STRESS_SCRIPT}"
print(f"Script encontrado: {STRESS_SCRIPT}")

# Obtener git hash
result = subprocess.run(["git", "rev-parse", "HEAD"],
                        capture_output=True, text=True, cwd="/content/tetris")
GIT_HASH = result.stdout.strip()
print(f"Git hash: {GIT_HASH}")

In [ ]:
# ============================================================
# PRUEBA RÁPIDA: 2 partidas con N_WORKERS — verificar que no se cuelga
# ============================================================
import time

out_dir_test = os.path.join(OUT_DIR_BASE, "out_stress_test")
os.makedirs(out_dir_test, exist_ok=True)

cmd_test = [
    sys.executable, STRESS_SCRIPT,
    "--n_games",       "2",
    "--max_pieces",    "100",
    "--tau",           str(TAU),
    "--board_value_weight", str(BVW),
    "--k",             str(K),
    "--H_min",         "4",
    "--H_max",         "15",
    "--p_bag_fill",    "1.0",
    "--alpha_depth",   str(ALPHA_DEPTH),
    "--n_workers",     str(N_WORKERS),
    "--out",           out_dir_test,
]

print(f">>> Prueba rápida (2 partidas, p_bag_fill=1.0, N_WORKERS={N_WORKERS})...")
t0 = time.time()

process = subprocess.Popen(
    cmd_test, cwd=CONF_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

TIMEOUT = 90
try:
    stdout, _ = process.communicate(timeout=TIMEOUT)
    elapsed = time.time() - t0
    print(stdout)
    print(f"OK — terminó en {elapsed:.1f}s (exit code: {process.returncode})")
    if process.returncode != 0:
        print("ERROR: exit code no-cero. Revisar output arriba.")
    else:
        print("Prueba superada — proceder con corrida diagnóstica.")
except subprocess.TimeoutExpired:
    process.kill()
    process.communicate()
    print(f"TIMEOUT ({TIMEOUT}s) — proceso colgado. Cambiar N_WORKERS=1 y reiniciar runtime.")

In [ ]:
# ============================================================
# BARRIDO p_bag_fill ∈ {0, 0.25, 0.5, 1.0}
# ============================================================
import time

sweep_results = {}  # p_bag_fill -> dict de bin_results

for p_fill in P_BAG_FILL_LEVELS:
    out_dir = os.path.join(OUT_DIR_BASE, f"out_stress_pbf{int(p_fill*100):03d}")
    os.makedirs(out_dir, exist_ok=True)

    cmd = [
        sys.executable, STRESS_SCRIPT,
        "--n_games",       str(N_GAMES),
        "--max_pieces",    str(MAX_PIECES),
        "--tau",           str(TAU),
        "--board_value_weight", str(BVW),
        "--k",             str(K),
        "--H_min",         "4",
        "--H_max",         "15",
        "--p_bag_fill",    str(p_fill),
        "--alpha_depth",   str(ALPHA_DEPTH),
        "--n_workers",     str(N_WORKERS),
        "--out",           out_dir,
    ]

    print(f"\n>>> p_bag_fill={p_fill} — iniciando...")
    t0 = time.time()

    process = subprocess.Popen(
        cmd, cwd=CONF_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in process.stdout:
        print(line, end="")
    rc = process.wait()

    elapsed = time.time() - t0
    print(f"Tiempo: {elapsed:.0f}s — exit code: {rc}")

    # Guardar git hash
    with open(os.path.join(out_dir, "git_hash.txt"), "w") as f:
        f.write(GIT_HASH + "\n")

    # Leer resultados
    json_path = os.path.join(out_dir, f"stress_results_k{K}.json")
    if os.path.exists(json_path):
        with open(json_path) as f:
            data = json.load(f)
        sweep_results[p_fill] = data["bin_results"]
        print(f"  Resultados cargados: {list(data['bin_results'].keys())}")
    else:
        print(f"  ADVERTENCIA: no se encontró {json_path}")

print("\nBarrido completo.")

In [ ]:
# ============================================================
# DIAGNÓSTICO DE INYECCIÓN — las dos correlaciones clave
# ============================================================
# Lee el CSV generado y calcula directamente:
#   c1 = corr(n_JL_fill_prev, depth_tras_inyectar)
#   c2 = corr(n_JL_fill_prev, depth_en_decision)
# Estas dos correlaciones determinan qué arreglar antes de la regresión.

import glob
from scipy import stats as scipy_stats

print("=" * 65)
print("DIAGNÓSTICO DE INYECCIÓN")
print("=" * 65)

for p_fill in P_BAG_FILL_LEVELS:
    csv_path = glob.glob(
        os.path.join(OUT_DIR_BASE, f"out_stress_pbf{int(p_fill*100):03d}",
                     "decisions_stress_k5.csv")
    )
    if not csv_path:
        print(f"pbf={p_fill}: CSV no encontrado")
        continue

    df_csv = pd.read_csv(csv_path[0])
    # Una fila por decisión (alternative_id=0 evita duplicar)
    df_d = df_csv[
        (df_csv["alternative_id"] == 0) &
        (df_csv["n_jl_fill_prev"] >= 0)
    ].copy()

    if len(df_d) < 20:
        print(f"pbf={p_fill}: muy pocas filas para diagnosticar ({len(df_d)})")
        continue

    c1, p1 = scipy_stats.pearsonr(df_d["n_jl_fill_prev"], df_d["depth_after_fill_prev"])
    c2, p2 = scipy_stats.pearsonr(df_d["n_jl_fill_prev"], df_d["board_depth_at_decision"])

    print(f"\np_bag_fill = {p_fill}  (n decisiones = {len(df_d)})")
    print(f"  c1 = corr(n_JL_fill_prev, depth_tras_inyectar) = {c1:+.4f}  (p={p1:.4f})")
    print(f"  c2 = corr(n_JL_fill_prev, depth_en_decision)   = {c2:+.4f}  (p={p2:.4f})")

    if abs(c1) < 0.05:
        diag = "CANAL CERRADO: la inyección no crea profundidad. Revisar bag_en_relleno_fill."
    elif abs(c1) > 0.05 and abs(c2) < 0.05:
        diag = "BORRADO: inyección funciona (c1>0) pero el fill la borra antes de la decisión (c2≈0)."
    elif abs(c2) > 0.05:
        diag = "CANAL VIVO: profundidad persiste hasta la decisión. La regresión tiene algo que medir."
    else:
        diag = "señal débil en ambas — diagnóstico ambiguo."

    print(f"  → {diag}")

print()
print("Interpretación:")
print("  Si BORRADO: hay que hacer que la profundidad inyectada persista")
print("  (lag-1 fiel a la estructura real del confound humano, no contemporánea).")
print("  Si CANAL CERRADO: revisar la lógica del generador.")

In [ ]:
# ============================================================
# TABLA: piso oracle por bin × p_bag_fill
# ============================================================
from tabulate import tabulate

H_BIN_LABELS = ["4-6", "7-8", "9-10", "11-12", "13-15"]

rows = []
for label in H_BIN_LABELS:
    row = [f"H={label}"]
    for p_fill in P_BAG_FILL_LEVELS:
        if p_fill not in sweep_results:
            row.append("n/d")
            continue
        bin_data = sweep_results[p_fill].get(label, {})
        ob = bin_data.get("oracle", {}).get("beta")
        op = bin_data.get("oracle", {}).get("pvalue")
        if ob is None:
            row.append("n/d")
        else:
            sig = "*" if (op is not None and op < 0.05) else ""
            row.append(f"{ob:+.3f}{sig}")
    rows.append(row)

headers = ["Bin"] + [f"pbf={p:.2f}" for p in P_BAG_FILL_LEVELS]
print("Piso oracle (β señal no-tracker) — * p<0.05")
print(tabulate(rows, headers=headers, tablefmt="grid"))
print()
print("Interpretación:")
print("  pbf=0.00 es el control puro (bag-ciego = esperado ≈ 0).")
print("  pbf=1.00 es la inyección máxima. Si algún bin muestra *, el oráculo falla.")

In [ ]:
# ============================================================
# COMPROBACIÓN DE MONOTONICIDAD
# ============================================================
# Si hay fallo real (no ruido), |β| debe crecer con p_bag_fill.
# Si es ruido: sin tendencia.

print("Tendencia de |β oracle| vs p_bag_fill por bin:\n")
all_pass = True

for label in H_BIN_LABELS:
    betas = []
    for p_fill in P_BAG_FILL_LEVELS:
        if p_fill not in sweep_results:
            betas.append(None)
            continue
        ob = sweep_results[p_fill].get(label, {}).get("oracle", {}).get("beta")
        betas.append(ob)

    valid = [(p, b) for p, b in zip(P_BAG_FILL_LEVELS, betas) if b is not None]
    if len(valid) < 2:
        print(f"  H={label}: datos insuficientes")
        continue

    ps, bs = zip(*valid)
    abs_bs = [abs(b) for b in bs]

    # Correlación de Spearman
    from scipy import stats as scipy_stats
    if len(valid) >= 3:
        rho, pval = scipy_stats.spearmanr(ps, abs_bs)
        direction = "↑ MONOTONO" if rho > 0.5 else ("↓ anti-monotono" if rho < -0.5 else "~ plano")
        if rho > 0.5 and pval < 0.2:  # umbral laxo dado n_levels=4
            all_pass = False
        print(f"  H={label}: {direction} (ρ={rho:+.2f}, p={pval:.2f})  abs_betas={[f'{b:.3f}' for b in abs_bs]}")
    else:
        delta = abs_bs[-1] - abs_bs[0]
        print(f"  H={label}: Δ|β|={delta:+.3f} (pbf 0→{valid[-1][0]})")

print()
if all_pass:
    print("VEREDICTO MONOTONICIDAD: no hay tendencia clara — posiblemente ruido, no fallo sistemático.")
else:
    print("VEREDICTO MONOTONICIDAD: |β| crece con p_bag_fill — fallo sistemático confirmado.")

In [ ]:
# ============================================================
# TEST DEL REMEDIO — --extend_oracle (solo si hay fallo)
# ============================================================
# Cambia FORCE_REMEDY = True para correr siempre.
FORCE_REMEDY = False

# Detectar fallo automáticamente
any_failure = any(
    sweep_results.get(p_fill, {}).get(label, {}).get("oracle", {}).get("pvalue", 1.0) < 0.05
    for p_fill in P_BAG_FILL_LEVELS
    for label in H_BIN_LABELS
)

if not any_failure and not FORCE_REMEDY:
    print("Sin fallo detectado — test del remedio no necesario.")
    print("(Ajustar FORCE_REMEDY=True para correrlo de todas formas.)")
else:
    P_FILL_REMEDY = 1.0  # intensidad maxima para el test del remedio
    out_dir_remedy = os.path.join(OUT_DIR_BASE, "out_stress_remedy")
    os.makedirs(out_dir_remedy, exist_ok=True)

    cmd = [
        sys.executable, STRESS_SCRIPT,
        "--n_games",       str(N_GAMES),
        "--max_pieces",    str(MAX_PIECES),
        "--tau",           str(TAU),
        "--board_value_weight", str(BVW),
        "--k",             str(K),
        "--H_min",         "4",
        "--H_max",         "15",
        "--p_bag_fill",    str(P_FILL_REMEDY),
        "--alpha_depth",   str(ALPHA_DEPTH),
        "--n_workers",     str(N_WORKERS),
        "--out",           out_dir_remedy,
        "--extend_oracle",
    ]

    print(">>> Test del remedio (--extend_oracle, p_bag_fill=1.0)...")
    process = subprocess.Popen(
        cmd, cwd=CONF_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in process.stdout:
        print(line, end="")
    rc = process.wait()
    print(f"exit code: {rc}")

    with open(os.path.join(out_dir_remedy, "git_hash.txt"), "w") as f:
        f.write(GIT_HASH + "\n")

    json_path = os.path.join(out_dir_remedy, f"stress_results_k{K}.json")
    if os.path.exists(json_path):
        with open(json_path) as f:
            remedy_data = json.load(f)
        remedy_bins = remedy_data["bin_results"]

        rows_r = []
        for label in H_BIN_LABELS:
            normal = sweep_results.get(P_FILL_REMEDY, {}).get(label, {}).get("oracle", {})
            remedy = remedy_bins.get(label, {}).get("oracle", {})
            nb, np_ = normal.get("beta"), normal.get("pvalue")
            rb, rp  = remedy.get("beta"), remedy.get("pvalue")
            ns = "*" if (np_ is not None and np_ < 0.05) else ""
            rs = "*" if (rp  is not None and rp  < 0.05) else ""
            rows_r.append([
                f"H={label}",
                f"{nb:+.3f}{ns}" if nb is not None else "n/d",
                f"{rb:+.3f}{rs}" if rb is not None else "n/d",
            ])

        print()
        print("Remedio: ¿añadir res_col_hole_depth cierra el piso?")
        print(tabulate(rows_r,
                       headers=["Bin", "β oracle normal", "β oracle+depth"],
                       tablefmt="grid"))
        print("(* p<0.05)")

In [ ]:
# ============================================================
# NOTA: la tabla y el veredicto de regresión son informativos
# solo si el diagnóstico de inyección confirma c2 >> 0.
# Con c2 ≈ 0, los β son ruido y no se concluye nada sobre el oráculo.
# ============================================================

failures_by_level = {}
for p_fill in P_BAG_FILL_LEVELS:
    failed_bins = [
        label
        for label in H_BIN_LABELS
        if sweep_results.get(p_fill, {}).get(label, {}).get("oracle", {}).get("pvalue", 1.0) < 0.05
    ]
    failures_by_level[p_fill] = failed_bins

print("=" * 60)
print("REGRESIÓN — piso oracle (informativo si c2 >> 0)")
print("=" * 60)

any_fail = any(len(v) > 0 for v in failures_by_level.values())

if not any_fail:
    print()
    print("Ningún bin alcanza p<0.05.")
    print("ATENCIÓN: esto es informativo solo si c2 >> 0 en el diagnóstico.")
    print("Si c2 ≈ 0, el canal no estaba activo y el resultado es no-informativo.")
else:
    print()
    print("FALLO detectado en:")
    for p_fill, bins in failures_by_level.items():
        if bins:
            print(f"  p_bag_fill={p_fill}: bins {bins}")
    print()
    print("El oráculo NO absorbe la covariación — verificar c2 para confirmar")
    print("que el canal estaba activo (no artefacto de N bajo).")

print()
print(f"Git hash: {GIT_HASH}")
print(f"N_GAMES={N_GAMES}, MAX_PIECES={MAX_PIECES}, ALPHA_DEPTH={ALPHA_DEPTH}")

In [ ]:
# ============================================================
# EMPAQUETAR RESULTADOS
# ============================================================
import shutil
from google.colab import files

zip_src = OUT_DIR_BASE
zip_dest = "/content/results_stress.zip"

shutil.make_archive("/content/results_stress", "zip", zip_src)
print(f"Zip creado: {zip_dest}")

files.download(zip_dest)
print("Descarga iniciada.")